# 01 — Document Loading

**Vai trò:** Data Engineer · **Task:** S2-DE-04 (Yêu cầu 9.1, 9.2)

Notebook này minh hoạ `DocumentLoader` (S1-DE-01/02): tải tài liệu `.txt`/`.md`/`.pdf`, kiểm tra `supports()`, xử lý lỗi (định dạng không hỗ trợ, file không tồn tại), và tải hàng loạt qua `load_directory()` — bước đầu tiên trong luồng `Load → Chunk → Embed → Store` của `RAGPipeline.index_document()`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import DocumentLoader

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\lh222k\AI-Research-Assistant-with-RAG


In [2]:
# Nội dung tài liệu mẫu — dùng để tạo file .txt/.md thật trong data/raw/
SAMPLE_TXT = 'Retrieval-Augmented Generation (RAG) la mot kien truc ket hop giua he thong\ntruy xuat thong tin (retrieval) va mo hinh sinh ngon ngu (generation). Thay vi\nchi dua vao kien thuc da hoc trong qua trinh huan luyen, mot he thong RAG se\ntim kiem cac doan van ban lien quan tu kho du lieu rieng truoc khi yeu cau LLM\nsoan cau tra loi - giup cau tra loi bam sat nguon tai lieu thuc te va giam\nhien tuong "ao giac" (hallucination).\n\nQuy trinh RAG co ban gom hai giai doan chinh: indexing (tai tai lieu, chia nho\nthanh chunk, tao embedding va luu vao vector store) va querying (embed cau hoi,\ntim cac chunk gan nhat ve mat ngu nghia, ghep thanh prompt va goi LLM sinh cau\ntra loi). Du an nay trien khai toan bo quy trinh do bang cac thanh phan chay\nhoan toan cuc bo thong qua OLLAMA, khong phu thuoc dich vu dam may.\n\nMoi vai tro ky su trong du an phu trach mot phan rieng: Data Engineer lo viec\ntai tai lieu, chia nho van ban va luu tru vector; Pipeline Engineer dieu phoi\nluong RAGPipeline, xay dung prompt va giao dien Streamlit; Model Engineer chiu\ntrach nhiem ket noi voi OLLAMA cho ca embedding lan sinh van ban. Su phoi hop\ngiua ba vai tro nay - dac biet o cac task phu thuoc lien vai tro - la diem mau\nchot de toan bo he thong hoat dong dung nhu thiet ke.\n'
SAMPLE_MD = '# Ghi chu: Chia nho van ban (Chunking) trong RAG\n\n## Vi sao can chia nho?\n\nTai lieu nguon thuong dai hon nhieu so voi gioi han ngu canh cua embedding\nmodel hoac LLM. Chia nho (chunking) giup:\n\n- Tao ra cac don vi van ban vua du de embedding bieu dien chinh xac ngu nghia.\n- Cho phep retrieval tra ve dung phan lien quan thay vi ca tai lieu dai.\n- Giam chi phi token khi dua context vao prompt cho LLM.\n\n## Cac chien luoc pho bien\n\n1. **Fixed-size** - cat theo so ky tu co dinh, don gian va de du doan kich\n   thuoc, nhung co the cat ngang giua cau hoac y.\n2. **Recursive** - chia theo thu tu uu tien doan van -> cau -> tu, giu duoc\n   ranh gioi tu nhien cua van ban tot hon fixed-size.\n3. **Semantic** - chia theo ranh gioi cau, huong toi cac chunk mach lac ve\n   mat ngu nghia.\n\n## Tham so quan trong\n\n`chunk_size` quyet dinh do dai toi da moi doan; `chunk_overlap` tao ra phan\nchong lap giua cac doan lien tiep de khong mat ngu canh o ranh gioi chunk -\nday la diem can thuc nghiem de tim cau hinh phu hop voi tai lieu cu the.\n'

## 1. Chuẩn bị tài liệu mẫu

`data/raw/` hiện còn trống — cell dưới đây tạo hai file mẫu (`.txt` và `.md`) để có dữ liệu thật cho các bước tiếp theo (và để notebook `02_text_chunking.ipynb` tái sử dụng). Việc ghi file là **idempotent** — chạy lại notebook không tạo bản sao.

In [3]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


# Chuẩn bị .txt và .md files
sample_files = {
    "sample_rag_overview.txt": SAMPLE_TXT,
    "sample_chunking_notes.md": SAMPLE_MD,
}

for name, text in sample_files.items():
    path = RAW_DIR / name
    if not path.exists():
        path.write_text(text, encoding="utf-8")
        print(f"Đã tạo: {path.relative_to(PROJECT_ROOT)}")
    else:
        print(f"Đã tồn tại: {path.relative_to(PROJECT_ROOT)}")

# Kiểm tra .pdf file (không bắt buộc — chỉ thêm vào nếu đã có sẵn)
pdf_name = "sample_pdf.pdf"
pdf_path = RAW_DIR / pdf_name
if pdf_path.exists():
    print(f"Đã tồn tại: {pdf_path.relative_to(PROJECT_ROOT)}")
    sample_files[pdf_name] = None  # file sẵn có, không cần ghi
else:
    print(f"Chưa có {pdf_path.name} — bỏ qua demo PDF (đặt file PDF vào data/raw/ để test)")


Đã tồn tại: data\raw\sample_rag_overview.txt
Đã tồn tại: data\raw\sample_chunking_notes.md
Chưa có sample_pdf.pdf — bỏ qua demo PDF (đặt file PDF vào data/raw/ để test)


## 2. Tải tài liệu `.txt`, `.md` và `.pdf`

`loader.load()` trả về `Document` với `doc_id` duy nhất (hash của đường dẫn tuyệt đối), `doc_type` được suy ra từ phần mở rộng, và `content` không rỗng.

In [4]:
loader = DocumentLoader()

for name in sample_files:
    doc = loader.load(str(RAW_DIR / name))
    print(f"{name}")
    print(f"  doc_id   = {doc.doc_id}")
    print(f"  doc_type = {doc.doc_type}")
    print(f"  content  = {len(doc.content)} ký tự")
    print(f"  preview  = {doc.content[:80]!r}...")
    print()

sample_rag_overview.txt
  doc_id   = 4c69f82b6454088f402448785cd72be8e63495ef7d862f3c9d3054c3bdef34db
  doc_type = DocumentType.TXT
  content  = 1255 ký tự
  preview  = 'Retrieval-Augmented Generation (RAG) la mot kien truc ket hop giua he thong\ntruy'...

sample_chunking_notes.md
  doc_id   = a6deecba4a341bf8110054e84d7fa5fd47e7fcf81618380b3ad586aca4c7876f
  doc_type = DocumentType.MARKDOWN
  content  = 1028 ký tự
  preview  = '# Ghi chu: Chia nho van ban (Chunking) trong RAG\n\n## Vi sao can chia nho?\n\nTai l'...



## 3. Kiểm tra `supports()` và xử lý lỗi (Yêu cầu 1.3, 1.4)

`DocumentLoader` chỉ hỗ trợ `.pdf`, `.txt`, `.md`/`.markdown`. Với định dạng không hỗ trợ hoặc file không tồn tại, `load()` ném lỗi mô tả rõ nguyên nhân thay vì để lộ traceback khó hiểu — notebook bắt các lỗi này để minh hoạ, đảm bảo chạy hết cell mà không phát sinh exception chưa xử lý (Yêu cầu 9.2).

In [5]:
candidates = [
    str(RAW_DIR / "sample_rag_overview.txt"),    # hỗ trợ
    str(RAW_DIR / "bao_cao.docx"),               # định dạng không hỗ trợ
    str(RAW_DIR / "khong_ton_tai.txt"),          # file không tồn tại
]

for path in candidates:
    supported = loader.supports(path)
    print(f"supports({Path(path).name!r}) = {supported}")
    try:
        doc = loader.load(path)
        print(f"  -> tải thành công: {len(doc.content)} ký tự")
    except Exception as exc:
        print(f"  -> lỗi (như mong đợi): {exc}")
    print()

supports('sample_rag_overview.txt') = True
  -> tải thành công: 1255 ký tự

supports('bao_cao.docx') = False
  -> lỗi (như mong đợi): Định dạng file .docx không được hỗ trợ

supports('khong_ton_tai.txt') = True
  -> lỗi (như mong đợi): [Errno 2] No such file or directory: 'D:\\lh222k\\AI-Research-Assistant-with-RAG\\data\\raw\\khong_ton_tai.txt'



## 4. Tải hàng loạt với `load_directory()`

Tải mọi tài liệu được hỗ trợ trong một thư mục cùng lúc — đây chính là cách `RAGPipeline.index_directory()` thu thập danh sách tài liệu cần index. Mỗi `Document.doc_id` phải duy nhất giữa nhiều file (Property 12).

In [6]:
documents = loader.load_directory(str(RAW_DIR))
doc_ids = [doc.doc_id for doc in documents]

print(f"Đã tải {len(documents)} tài liệu từ {RAW_DIR.relative_to(PROJECT_ROOT)}:")
for doc in documents:
    name = Path(doc.file_path).name
    print(f"  - {name:30s} | doc_id={doc.doc_id} | {len(doc.content):5d} ký tự")

assert len(doc_ids) == len(set(doc_ids)), "doc_id phải duy nhất giữa các tài liệu!"
print("\ndoc_id duy nhất giữa mọi tài liệu: OK")

Đã tải 4 tài liệu từ data\raw:
  - sample_chunking_notes.md       | doc_id=a6deecba4a341bf8110054e84d7fa5fd47e7fcf81618380b3ad586aca4c7876f |  1028 ký tự
  - sample_rag_overview.txt        | doc_id=4c69f82b6454088f402448785cd72be8e63495ef7d862f3c9d3054c3bdef34db |  1255 ký tự
  - System Design Architecture cho Roadmap Product.xmind.pdf | doc_id=bddd8f63ab188deb6a1f8cf7fc0955e9d5bbad9400ade17c446722d99512d8bf |     0 ký tự
  - Trương Lê Huy - Numerology.pdf | doc_id=fe02ca94c0a940a7d991e69bb94ce51eaf7bf2badd042322374d3b0a581aeb57 |     0 ký tự

doc_id duy nhất giữa mọi tài liệu: OK


## 5. Tổng kết

- `DocumentLoader` tải thật `.txt`/`.md`/`.pdf`, sinh `doc_id` duy nhất và suy ra đúng `DocumentType` từ phần mở rộng file.
- Định dạng không hỗ trợ và file không tồn tại đều trả về lỗi mô tả rõ nguyên nhân — không làm sập notebook hay pipeline.
- `load_directory()` cho phép `RAGPipeline.index_directory()` xử lý hàng loạt tài liệu — đây là bước **Load** đầu tiên trong luồng `index_document()`.
- Hai tài liệu mẫu vừa tạo (`sample_rag_overview.txt`, `sample_chunking_notes.md`) sẽ được tái sử dụng ở notebook tiếp theo: `02_text_chunking.ipynb`.